# Generative Market Simulation -- Demo

**EECS 4904 -- Spring 2026**

This notebook demonstrates the full pipeline:
1. Download and preprocess real financial data
2. Train generative models (DDPM, TimeGAN, VAE, GARCH, Normalizing Flow)
3. Generate synthetic return paths
4. Validate against 6 stylized facts
5. Compare models side-by-side

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch

sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

## 1. Data Download & Preprocessing

In [ ]:
from src.data.download import download_market_data
from src.data.preprocess import prepare_dataset

DATA_DIR = os.path.join('..', 'data')

# Download (only needs to run once)
prices = download_market_data(start='2005-01-01', output_dir=DATA_DIR)

In [ ]:
# Preprocess into windows
dataset = prepare_dataset(data_dir=DATA_DIR, window_size=60, stride=5)

returns_df = dataset['returns_df']
windows = dataset['windows']
scaler = dataset['scaler_params']
asset_names = dataset['asset_names']
window_dates = dataset['window_dates']

print(f'Returns:  {returns_df.shape}')
print(f'Windows:  {windows.shape}')
print(f'Assets:   {asset_names[:5]} ... ({len(asset_names)} total)')

# Compute regime labels and conditioning vectors
from src.data.regime_labels import prepare_regime_data

regime_data = prepare_regime_data(
    returns_df=returns_df,
    vix_df=dataset.get('vix'),
    macro_df=dataset.get('macro_df'),
    window_dates=window_dates,
    data_dir=DATA_DIR,
)

window_cond = regime_data['window_cond']
window_regimes = regime_data['window_regimes']
print(f'Conditioning vectors: {window_cond.shape}')
print(f'Regime labels: {np.bincount(window_regimes)} (normal/crisis/calm)')

## 2. Validate Real Data (Sanity Check)

Before testing models, confirm real data passes all 6 stylized fact tests.

In [ ]:
from src.evaluation.stylized_facts import run_all_tests, print_report

real_returns = returns_df.values.astype(np.float32)
real_results = run_all_tests(real_returns)
print_report(real_results)

## 3. Train Models

Each model is trained on the windowed return data. Reduce `epochs` for a quick test.

In [ ]:
QUICK = True  # Set False for full training
EPOCHS = 20 if QUICK else 200
N_SAMPLES = 500 if QUICK else 2000
BATCH_SIZE = 32
COND_DIM = window_cond.shape[1] if window_cond is not None else 0

In [ ]:
# --- GARCH (fastest, no GPU needed) ---
from src.models.garch import GARCHModel

garch = GARCHModel(n_features=len(asset_names))
garch.train(real_returns)
garch_syn = garch.generate(N_SAMPLES, seq_len=60)
print(f'GARCH synthetic: {garch_syn.shape}')

In [ ]:
# --- VAE ---
from src.models.vae import FinancialVAE

vae = FinancialVAE(n_features=len(asset_names), seq_len=60, device=DEVICE)
vae.train(windows, epochs=EPOCHS, batch_size=BATCH_SIZE)
vae_syn = vae.generate(N_SAMPLES)
print(f'VAE synthetic: {vae_syn.shape}')

In [ ]:
# --- TimeGAN ---
from src.models.gan import TimeGANModel

tgan = TimeGANModel(n_features=len(asset_names), seq_len=60, device=DEVICE)
tgan.train(windows, epochs=EPOCHS, batch_size=BATCH_SIZE)
tgan_syn = tgan.generate(N_SAMPLES)
print(f'TimeGAN synthetic: {tgan_syn.shape}')

In [ ]:
# --- DDPM (with conditional generation) ---
from src.models.ddpm import DDPMModel

ddpm = DDPMModel(
    n_features=len(asset_names), seq_len=60,
    T=200 if QUICK else 1000,
    base_channels=32 if QUICK else 64,
    cond_dim=COND_DIM,
    device=DEVICE,
)
ddpm.train(windows, cond=window_cond, epochs=EPOCHS, batch_size=BATCH_SIZE)
ddpm_syn = ddpm.generate(N_SAMPLES)
print(f'DDPM synthetic: {ddpm_syn.shape}')

In [ ]:
# --- Normalizing Flow ---
from src.models.normalizing_flow import NormalizingFlowModel

flow = NormalizingFlowModel(
    n_features=len(asset_names), seq_len=60,
    hidden_dim=128, n_flow_layers=4,
    device=DEVICE,
)
flow.train(windows, epochs=EPOCHS, batch_size=BATCH_SIZE)
flow_syn = flow.generate(N_SAMPLES)
print(f'Flow synthetic: {flow_syn.shape}')

## 4. Stylized Facts Validation

In [ ]:
all_synthetic = {
    'GARCH': garch_syn,
    'VAE': vae_syn,
    'TimeGAN': tgan_syn,
    'DDPM': ddpm_syn,
    'NormFlow': flow_syn,
}

all_results = {}
for name, syn in all_synthetic.items():
    flat = syn.reshape(-1, syn.shape[-1]) if syn.ndim == 3 else syn
    results = run_all_tests(flat)
    all_results[name] = results
    n_pass = sum(1 for r in results if r.get('pass'))
    print(f'{name:12s}: {n_pass}/6 stylized facts passed')

## 5. Comparison Visualizations

In [ ]:
from src.evaluation.visualization import (
    plot_return_distributions,
    plot_qq_comparison,
    plot_acf_comparison,
    plot_stylized_facts_heatmap,
    plot_synthetic_paths,
)

real_flat = real_returns.flatten()
syn_flat = {name: s.flatten() for name, s in all_synthetic.items()}

In [ ]:
# Return distributions
fig = plot_return_distributions(real_flat, syn_flat)
plt.show()

In [ ]:
# QQ plots
fig = plot_qq_comparison(real_flat, syn_flat)
plt.show()

In [ ]:
# ACF of absolute returns
fig = plot_acf_comparison(real_flat, syn_flat, mode='absolute')
plt.show()

In [ ]:
# Stylized facts heatmap
fig = plot_stylized_facts_heatmap(all_results)
plt.show()

In [ ]:
# Synthetic paths (DDPM)
fig = plot_synthetic_paths(ddpm_syn, n_paths=50, asset_idx=0, title='DDPM')
plt.show()

## 6. Quantitative Comparison

In [ ]:
from src.evaluation.metrics import full_evaluation

comparison_rows = []
for name, syn in all_synthetic.items():
    metrics = full_evaluation(real_returns[:2000], syn[:2000])
    n_pass = sum(1 for r in all_results[name] if r.get('pass'))
    comparison_rows.append({
        'Model': name,
        'MMD': f'{metrics["mmd"]:.6f}',
        'Wasserstein': f'{metrics["wasserstein_1d"]:.6f}',
        'KS Stat': f'{metrics["ks_stat"]:.4f}',
        'Disc. Score': f'{metrics["discriminative_score"]:.2f}',
        'Corr. Dist.': f'{metrics["correlation_matrix_distance"]:.2f}',
        'Kurt Diff': f'{metrics["moments"]["abs_diff"]["kurtosis"]:.4f}',
        'SF Passed': f'{n_pass}/6',
    })

pd.DataFrame(comparison_rows).set_index('Model')

## 7. Conditional Generation (DDPM)

The DDPM is trained with macro regime conditioning (`cond_dim > 0`).
We can generate crisis, calm, and normal scenarios using the learned conditioning vectors.

In [ ]:
# Generate crisis vs calm vs normal scenarios with regime conditioning
from src.data.regime_labels import get_regime_conditioning_vectors

regime_vectors = get_regime_conditioning_vectors()

crisis_syn = ddpm.generate(200, seq_len=60, cond=regime_vectors['crisis'])
calm_syn = ddpm.generate(200, seq_len=60, cond=regime_vectors['calm'])
normal_syn = ddpm.generate(200, seq_len=60, cond=regime_vectors['normal'])

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

configs = [
    (crisis_syn, 'Crisis', '#E53935'),
    (calm_syn, 'Calm', '#43A047'),
    (normal_syn, 'Normal', '#455A64'),
]

for ax, (syn, label, color) in zip(axes, configs):
    for i in range(50):
        cum = np.exp(np.cumsum(syn[i, :, 0])) * 100
        ax.plot(cum, alpha=0.3, color=color, lw=0.8)
    terminal = np.exp(np.cumsum(syn[:, :, 0], axis=1))[:, -1] * 100
    ax.set_title(f'{label} Regime\nMean: ${np.mean(terminal):.1f} | VaR: ${np.percentile(terminal, 5):.1f}',
                 fontweight='bold')
    ax.set_xlabel('Trading Day')
    ax.set_ylabel('Price (start=100)')

fig.suptitle('Conditional Generation: Regime Scenarios (DDPM)', fontsize=14, fontweight='bold')
fig.tight_layout()
plt.show()

---

**End of Demo.** For full results:
- Set `QUICK = False` and `EPOCHS = 200`
- Run the full pipeline: `python -m src.run_pipeline`
- Start the interactive demo: `python -m src.demo.app`